# Vector DB Creation

In [ ]:
PINECONE_API_KEY=""

In [3]:
from pinecone import Pinecone, ServerlessSpec

# Initialize a client
pc = Pinecone(api_key=PINECONE_API_KEY)

### Setting up the embedding model (dense)

In [4]:
import torch

# Check and set device
device = torch.device("mps") if torch.backends.mps.is_available() else "cpu"

print(f"Using device: {device}")

Using device: mps


In [5]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
embed_model.to(device)

/Users/shawn.n/Desktop/Deen/vector_db_setup/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SentenceTransformer(
  (0): Transformer({'max_seq_length': 384, 'do_lower_case': False}) with Transformer model: MPNetModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [6]:
# Encode example sentences
example_sentences = ["This is an example sentence", "Each sentence is converted"]
example_embeddings = embed_model.encode(example_sentences, device=str(device))  # Perform encoding on GPU

print("Embeddings generated successfully!")
print(example_embeddings)

Embeddings generated successfully!
[[ 0.0225026  -0.07829171 -0.02303075 ... -0.00827928  0.02652684
  -0.00201899]
 [ 0.0417024   0.00109742 -0.01553416 ... -0.0218163  -0.06359359
  -0.00875286]]


### Embedding the data and upserting to vector db

In [7]:
# Create index

index_name = "deen-index-1"

pc.create_index(
    name=index_name,
    dimension=768, # Replace with your model dimensions
    metric="cosine", # Replace with your model metric
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    ) 
)

In [73]:
import pandas as pd

# Load your dataset
df = pd.read_csv("../cleaned_csv_files/alkafi_vol4_cleaned.csv")  # Replace with your CSV file path


#### Documents for al kafi

In [74]:
# Prepare data for embeddings
documents = [
    f"Source: Al Kafi | Volume: {row['volume']} | Book: {row['book']} | Chapter: {row['chapter']} | Hadith number: {row['hadees_number']} | Text: {row['hadees_english']}"
    for _, row in df.iterrows()
]

In [75]:
# Generate embeddings
embeddings = embed_model.encode(documents, batch_size=32, show_progress_bar=True)

print("Embeddings generated!")

Batches: 100%|██████████| 69/69 [00:17<00:00,  3.96it/s]


Embeddings generated!


In [70]:
# Wait for the index to be ready
while not pc.describe_index(index_name).status['ready']:
    time.sleep(1)

index = pc.Index(index_name)

In [76]:
vectors = []
for i in range(len(embeddings)):
    vectors.append({
        "id": f"alkafi_vol4_vec{i}",
        "values": embeddings[i],
        "metadata": {"text": df.iloc[i]["hadees_english"],"source": df.iloc[i]["title"], 'author': df.iloc[i]['author'], 'volume': df.iloc[i]['volume'], 'chapter': df.iloc[i]["chapter"], 'hadith_number': str(df.iloc[i]["hadees_number"]), 'book': df.iloc[i]["book"], 'chapter': df.iloc[i]["chapter"]}
    })

In [77]:
BATCH_SIZE = 100  # Adjust based on the size of your embeddings and metadata

# Split data into batches
for i in range(0, len(vectors), BATCH_SIZE):
    batch = vectors[i:i + BATCH_SIZE]  # Get a batch of vectors
    index.upsert(vectors=batch, namespace="ns1")  # Upsert the batch
    print(f"Batch {i // BATCH_SIZE + 1} upserted successfully!")

Batch 1 upserted successfully!
Batch 2 upserted successfully!
Batch 3 upserted successfully!
Batch 4 upserted successfully!
Batch 5 upserted successfully!
Batch 6 upserted successfully!
Batch 7 upserted successfully!
Batch 8 upserted successfully!
Batch 9 upserted successfully!
Batch 10 upserted successfully!
Batch 11 upserted successfully!
Batch 12 upserted successfully!
Batch 13 upserted successfully!
Batch 14 upserted successfully!
Batch 15 upserted successfully!
Batch 16 upserted successfully!
Batch 17 upserted successfully!
Batch 18 upserted successfully!
Batch 19 upserted successfully!
Batch 20 upserted successfully!
Batch 21 upserted successfully!
Batch 22 upserted successfully!


### Nahjul Balaghah

In [96]:
import pandas as pd

# Load your dataset
df = pd.read_csv("../cleaned_csv_files/cleaned_nahjulbalagha2.csv")  # Replace with your CSV file path


In [97]:
# Prepare data for embeddings
documents = [
    f"Source: Nahj al-Balaghah | Section: {row['book']} | {row['chapter']} | Text: {row['hadees_english']}"
    for _, row in df.iterrows()
]

In [98]:
# Generate embeddings
embeddings = embed_model.encode(documents, batch_size=32, show_progress_bar=True)

print("Embeddings generated!")

Batches: 100%|██████████| 71/71 [00:13<00:00,  5.33it/s]

Embeddings generated!


In [99]:
# Wait for the index to be ready
while not pc.describe_index(index_name).status['ready']:
    time.sleep(1)

index = pc.Index(index_name)

In [100]:
vectors = []
for i in range(len(embeddings)):
    vectors.append({
        "id": f"nahjulbalagha_vec{i}",
        "values": embeddings[i],
        "metadata": {"text": df.iloc[i]["hadees_english"],"source": "Nahj al-Balaghah", 'author': "Imam Ali ibn Abu Talib", 'volume': "NA", 'chapter': df.iloc[i]["chapter"], 'hadith_number': str(df.iloc[i]["hadees_number"]), 'book': df.iloc[i]["book"]}
    })

In [101]:
BATCH_SIZE = 100  # Adjust based on the size of your embeddings and metadata

# Split data into batches
for i in range(0, len(vectors), BATCH_SIZE):
    batch = vectors[i:i + BATCH_SIZE]  # Get a batch of vectors
    index.upsert(vectors=batch, namespace="ns1")  # Upsert the batch
    print(f"Batch {i // BATCH_SIZE + 1} upserted successfully!")

Batch 1 upserted successfully!
Batch 2 upserted successfully!
Batch 3 upserted successfully!
Batch 4 upserted successfully!
Batch 5 upserted successfully!
Batch 6 upserted successfully!
Batch 7 upserted successfully!
Batch 8 upserted successfully!
Batch 9 upserted successfully!
Batch 10 upserted successfully!
Batch 11 upserted successfully!
Batch 12 upserted successfully!
Batch 13 upserted successfully!
Batch 14 upserted successfully!
Batch 15 upserted successfully!
Batch 16 upserted successfully!
Batch 17 upserted successfully!
Batch 18 upserted successfully!
Batch 19 upserted successfully!
Batch 20 upserted successfully!
Batch 21 upserted successfully!
Batch 22 upserted successfully!
Batch 23 upserted successfully!


### For deleting records based on id prefix

In [93]:
index = pc.Index("deen-index-1")

In [95]:

for ids in index.list(prefix='nahjulbalagha', namespace='ns1'):
  print(ids) # ['doc1#chunk1', 'doc1#chunk2', 'doc1#chunk3']
  index.delete(ids=ids, namespace='ns1')

['nahjulbalagha_vec2168', 'nahjulbalagha_vec2169', 'nahjulbalagha_vec217', 'nahjulbalagha_vec2170', 'nahjulbalagha_vec2171', 'nahjulbalagha_vec2172', 'nahjulbalagha_vec2173', 'nahjulbalagha_vec2174', 'nahjulbalagha_vec2175', 'nahjulbalagha_vec2176', 'nahjulbalagha_vec2177', 'nahjulbalagha_vec2178', 'nahjulbalagha_vec2179', 'nahjulbalagha_vec218', 'nahjulbalagha_vec2180', 'nahjulbalagha_vec2181', 'nahjulbalagha_vec2182', 'nahjulbalagha_vec2183', 'nahjulbalagha_vec2184', 'nahjulbalagha_vec2185', 'nahjulbalagha_vec2186', 'nahjulbalagha_vec2187', 'nahjulbalagha_vec2188', 'nahjulbalagha_vec2189', 'nahjulbalagha_vec219', 'nahjulbalagha_vec2190', 'nahjulbalagha_vec2191', 'nahjulbalagha_vec2192', 'nahjulbalagha_vec2193', 'nahjulbalagha_vec2194', 'nahjulbalagha_vec2195', 'nahjulbalagha_vec2196', 'nahjulbalagha_vec2197', 'nahjulbalagha_vec2198', 'nahjulbalagha_vec2199', 'nahjulbalagha_vec22', 'nahjulbalagha_vec220', 'nahjulbalagha_vec2200', 'nahjulbalagha_vec2201', 'nahjulbalagha_vec2202', 'nahj

## Sahih Bukhari

In [8]:
import pandas as pd

# Load your dataset
df = pd.read_csv("../cleaned_csv_files/sunni/sahih_bukhari.csv")  # Replace with your CSV file path


In [9]:
# Prepare data for embeddings
documents = [
    f"Source: Sahih Bukhari | Chapter Number: {row['chapter_no']} | Chapter: {row['chapter']} | Text: {row['text_en']}"
    for _, row in df.iterrows()
]

In [10]:
# Generate embeddings
embeddings = embed_model.encode(documents, batch_size=32, show_progress_bar=True)

print("Embeddings generated!")

Batches: 100%|██████████| 225/225 [00:54<00:00,  4.13it/s]

Embeddings generated!


In [11]:
index_name = "deen-index-sunni-1"
# Wait for the index to be ready
while not pc.describe_index(index_name).status['ready']:
    time.sleep(1)

index = pc.Index(index_name)


In [13]:
vectors = []
for i in range(len(embeddings)):
    vectors.append({
        "id": f"sahihbukhari_vec{i}",
        "values": embeddings[i],
        "metadata": {"text": df.iloc[i]["text_en"], "source": "Sahih Bukhari", 'author': "Mohammed Al-Bukhari", 'volume': "NA", 'chapter': f"{df.iloc[i]['chapter_no']} - {df.iloc[i]['chapter']}", 'hadith_number': str(df.iloc[i]["hadith_no"]), 'book': "NA"}
    })

In [14]:
BATCH_SIZE = 100  # Adjust based on the size of your embeddings and metadata

# Split data into batches
for i in range(0, len(vectors), BATCH_SIZE):
    batch = vectors[i:i + BATCH_SIZE]  # Get a batch of vectors
    index.upsert(vectors=batch, namespace="ns1")  # Upsert the batch
    print(f"Batch {i // BATCH_SIZE + 1} upserted successfully!")

Batch 1 upserted successfully!
Batch 2 upserted successfully!
Batch 3 upserted successfully!
Batch 4 upserted successfully!
Batch 5 upserted successfully!
Batch 6 upserted successfully!
Batch 7 upserted successfully!
Batch 8 upserted successfully!
Batch 9 upserted successfully!
Batch 10 upserted successfully!
Batch 11 upserted successfully!
Batch 12 upserted successfully!
Batch 13 upserted successfully!
Batch 14 upserted successfully!
Batch 15 upserted successfully!
Batch 16 upserted successfully!
Batch 17 upserted successfully!
Batch 18 upserted successfully!
Batch 19 upserted successfully!
Batch 20 upserted successfully!
Batch 21 upserted successfully!
Batch 22 upserted successfully!
Batch 23 upserted successfully!
Batch 24 upserted successfully!
Batch 25 upserted successfully!
Batch 26 upserted successfully!
Batch 27 upserted successfully!
Batch 28 upserted successfully!
Batch 29 upserted successfully!
Batch 30 upserted successfully!
Batch 31 upserted successfully!
Batch 32 upserted